# 08 · Geocode the NT school list

**Replaces** `geocode_schools.py`.

The NT Department of Education school list has no coordinates. Each school gets one from, in order:

1. **ACARA School Location 2025**: the school's own site coordinate (`school_geocode_precision = site`).
   Matched on the school name, or through a short cited alias list where the same school appears
   under a newer name or as a campus.
2. **Stand-alone preschools** are not in ACARA. They take the ACARA point of the school with the same
   name (`Alawa Pre School` → `Alawa Primary School`), because NT preschools almost always share that
   school's grounds (`school_geocode_precision = sibling_school`).
3. Anything still unmatched falls back to its **locality** (suburb or community) point from a local
   gazetteer, exactly as before (`suburb` / `community` / `region` / `none`).

**Reads**

| file | used for |
|---|---|
| `data/external/services/trimmed/school_list_trimmed.csv` | the 273 schools, their locality, region, sector and stage flags |
| `data/external/services/acara/School Location 2025.xlsx` | ACARA site coordinates (NT rows) |
| `data/processed/medical_services.csv` (from nb 07) | suburb centroids (locality fallback) |
| `data/processed/emergency_services.csv` (from nb 07) | suburb centroids (locality fallback) |
| `data/raw/bushtel/Com_BushTel_Profile_CMC_2024.json` | 792 NT communities and their aliases (locality fallback) |

**Writes** `data/processed/school_services.csv`.

**Fixes compared with the old script**

1. **Site coordinates from ACARA** instead of one shared point per town (e.g. the 27 Alice Springs
   schools no longer all sit on the same point).
2. **Katherine schools were ~57 km out of place.** Suburb centroids were a plain mean, and the
   `Katherine` suburb contains clinics from Ngukurr, Minyerri and Barunga. Rows flagged in nb 07
   are now left out of centroids.
3. **Postcodes stuck to the locality** (`Wulagi 0812`, `Holtze 0829`) are stripped, and
   placeholders (`NT`, `tba`) are treated as missing.
4. **Darwin suburbs no longer collapse onto the Darwin centre.** Alawa, Anula, Nakara, Wanguri
   and others use their own Wikipedia coordinates (cited per row) when the locality fallback is needed.
5. **More schools are resolved.** The fallback tries a shorter locality
   (`Pickertaramoor Melville Island` → `Pickertaramoor`), then a BushTel community named in the
   school's own name (`Nawarddeken Academy Manmoyi School` → Manmoyi).
6. **`school_geocode_precision` column** (`site` / `sibling_school` / `suburb` / `community` /
   `region` / `none`), so anything downstream can tell a real site from a coarse fallback.
7. A school with no region is filled from other schools in the same locality.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Walk up from wherever Jupyter was started until we find the repo root
# (the folder that holds data/raw), so the notebook runs from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INPUTS = ROOT / "data" / "external" / "services" / "trimmed"   # read-only inputs
PROCESSED = ROOT / "data" / "processed"                         # the only folder we write to

# NT bounding box used as a sanity check on every coordinate we write.
NT_LAT, NT_LON = (-26.1, -10.9), (128.9, 138.1)


def in_nt(df: pd.DataFrame) -> pd.Series:
    return df["latitude"].between(*NT_LAT) & df["longitude"].between(*NT_LON)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorised; any argument may be an array)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

import json
import re

BUSHTEL_JSON = ROOT / "data" / "raw" / "bushtel" / "Com_BushTel_Profile_CMC_2024.json"
ACARA_XLSX = ROOT / "data" / "external" / "services" / "acara" / "School Location 2025.xlsx"


## 1. Load the school list

In [2]:
schools = pd.read_csv(INPUTS / "school_list_trimmed.csv")
assert schools["name"].is_unique
print(f"{len(schools)} schools")
schools.head()


273 schools


,source,name,school_locality,school_region,school_sector,school_remoteness,school_is_preschool,school_is_primary,school_is_middle,school_is_senior,school_is_faft
0,school,Acacia Hill School,Alice Springs,Central,Government,Urban,True,True,True,True,False
1,school,Adelaide River School,Adelaide River,Top End,Government,Remote Special Category,True,True,False,False,False
2,school,Alawa Pre School,Alawa,Darwin,NaN,NaN,True,False,False,False,False
3,school,Alawa Primary School,Alawa,Darwin,Government,Urban,True,True,False,False,False
4,school,Alcoota School,Engawala,Central,Government,Remote Category 2,True,True,True,True,False


## 2. Clean the locality field

The locality was parsed from a free-text address, so a few values still carry a postcode or are
placeholders rather than place names.

In [3]:
PLACEHOLDERS = {"", "NT", "N/A", "TBA"}


def norm(name) -> str:
    """Normalise a place name for matching: upper-case, single spaces, no trailing '(NT)'."""
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    n = re.sub(r"\s+", " ", str(name).strip().upper())
    return re.sub(r"\s*\(NT\)$", "", n)


def clean_locality(raw):
    """Strip a trailing postcode and turn placeholders into None."""
    if pd.isna(raw):
        return None
    loc = re.sub(r"\s+\d{4}$", "", str(raw).strip())      # 'Wulagi 0812' -> 'Wulagi'
    return None if loc.upper() in PLACEHOLDERS else loc


schools["school_locality_raw"] = schools["school_locality"]
schools["school_locality"] = schools["school_locality"].map(clean_locality)

changed = schools[schools["school_locality_raw"].fillna("") != schools["school_locality"].fillna("")]
print(changed[["name", "school_locality_raw", "school_locality"]].to_string())


                                         name             school_locality_raw            school_locality
135                   Mapuru Yirralka College                         NT 0822                       None
162  Nawarddeken Academy Kabulwarnamyo School                             tba                       None
163      Nawarddeken Academy Kunmayali School                             tba                       None
164    Nawarddeken Academy Mamadawerre School                             tba                       None
165        Nawarddeken Academy Manmoyi School                             tba                       None
181           Northern Territory Music School                     Wulagi 0812                     Wulagi
217              School of Sport Education NT                              NT                       None
224              St Joseph's Catholic College                  Katherine 0850                  Katherine
230                           Stirling School  Wilora (

## 3. Build the gazetteer (place name → coordinate)

Sources in priority order (the first source to supply a name wins):

1. **Medical suburb centroids.** Mean of the facilities in that suburb, **excluding** rows nb 07
   flagged as `far_from_site`.
2. **Emergency suburb centroids.** Same rule.
3. **BushTel communities and their aliases.** One point per remote community.

Suburb centroids come first because a suburb's facilities sit inside the suburb. BushTel gives
one point per town, which for a big town like Alice Springs is less representative.

In [4]:
med = pd.read_csv(PROCESSED / "medical_services.csv")
emg = pd.read_csv(PROCESSED / "emergency_services.csv")

gaz: dict[str, tuple[float, float, str]] = {}      # NAME -> (lat, lon, source tag)


def add_gaz(name, lat, lon, source):
    key = norm(name)
    if key and pd.notna(lat) and pd.notna(lon) and key not in gaz:
        gaz[key] = (float(lat), float(lon), source)


# 3a/3b: suburb centroids from the facility files, leaving out mislabelled rows.
for df, suburb_col, flag_col, tag in [
    (med, "medical_suburb", "medical_suburb_far_from_site", "medical_suburb"),
    (emg, "emergency_suburb", "emergency_suburb_far_from_site", "emergency_suburb"),
]:
    good = df[~df[flag_col]]
    centroids = good.groupby(good[suburb_col].map(norm))[["latitude", "longitude"]].mean()
    for suburb, row in centroids.iterrows():
        add_gaz(suburb, row.latitude, row.longitude, tag)

# 3c: BushTel communities, then every recorded alias.
bushtel = json.loads(BUSHTEL_JSON.read_text(encoding="utf-8"))["features"]
bushtel_keys = set()
for feat in bushtel:
    p = feat["properties"]
    names = [p["community_name"]]
    aliases = p.get("community_aliases") or ""
    if "no aliases" not in aliases.lower():
        names += [a for a in re.split(r"[;,]", aliases) if a.strip()]
    for i, n in enumerate(names):
        add_gaz(n, p["latitude"], p["longitude"], "bushtel" if i == 0 else "bushtel_alias")
        bushtel_keys.add(norm(n))

print(f"gazetteer: {len(gaz)} names ->", pd.Series([v[2] for v in gaz.values()]).value_counts().to_dict())
print("KATHERINE now:", gaz["KATHERINE"])


gazetteer: 2156 names -> {'bushtel_alias': 1301, 'bushtel': 712, 'medical_suburb': 101, 'emergency_suburb': 42}
KATHERINE now: (-14.456183343590547, 132.26592077988724, 'medical_suburb')


## 4. Manual overrides (cited)

Darwin, Palmerston and rural-Darwin suburbs that have no facility in the medical/emergency files
and no BushTel entry. Each was looked up individually, and the source is stored with the row.
These **take priority** over the gazetteer. They're checked against the NT box below.

In [5]:
MANUAL_OVERRIDES = {
    # suburb          lat          lon          source
    "ALAWA":         (-12.380000, 130.873000, "wikipedia:Alawa,_Northern_Territory"),
    "ANULA":         (-12.392778, 130.890000, "wikipedia:Anula,_Northern_Territory"),
    "JINGILI":       (-12.386667, 130.875278, "wikipedia:Jingili,_Northern_Territory"),
    "NAKARA":        (-12.369444, 130.878056, "wikipedia:Nakara,_Northern_Territory"),
    "RAPID CREEK":   (-12.380833, 130.862222, "wikipedia:Rapid_Creek,_Northern_Territory"),
    "THE GARDENS":   (-12.448611, 130.836389, "wikipedia:The_Gardens,_Northern_Territory"),
    "LARRAKEYAH":    (-12.458611, 130.821667, "wikipedia:Larrakeyah,_Northern_Territory"),
    "WAGAMAN":       (-12.382500, 130.885000, "wikipedia:Wagaman,_Northern_Territory"),
    "WANGURI":       (-12.372500, 130.887778, "wikipedia:Wanguri,_Northern_Territory"),
    "WULAGI":        (-12.384444, 130.893056, "wikipedia:Wulagi,_Northern_Territory"),
    "HOLTZE":        (-12.461000, 130.984000, "wikipedia:Holtze,_Northern_Territory"),
    "FREDS PASS":    (-12.539836, 131.055036, "wikipedia:Freds_Pass,_Northern_Territory"),
    "MARLOW LAGOON": (-12.498405, 130.964515, "wikipedia:Marlow_Lagoon,_Northern_Territory"),
    "BEES CREEK":    (-12.555285, 131.059105, "wikipedia:Bees_Creek,_Northern_Territory"),
    "DRIVER":        (-12.485085, 130.973005, "wikipedia:Driver,_Northern_Territory"),
    "GIRRAWEEN":     (-12.524085, 131.092505, "wikipedia:Girraween,_Northern_Territory"),
    "GRAY":          (-12.492205, 130.978915, "wikipedia:Gray,_Northern_Territory"),
    "MOULDEN":       (-12.510000, 130.976000, "wikipedia:Moulden,_Northern_Territory"),
    "JOHNSTON":      (-12.486400, 131.009600, "wikipedia:Johnston,_Northern_Territory"),
    "DUNDEE BEACH":  (-12.733400, 130.383100, "wikipedia:Dundee_Beach,_Northern_Territory"),
    "MIDDLE POINT":  (-12.599000, 131.326100, "wikipedia:Middle_Point,_Northern_Territory"),
    "BELLAMACK":     (-12.521100, 130.980200, "latitude.to:Bellamack"),
    "TIPPERARY":     (-13.729839, 131.040459, "findlatitudeandlongitude.com:Tipperary_Station"),
}
for k, (lat, lon, _) in MANUAL_OVERRIDES.items():
    assert NT_LAT[0] <= lat <= NT_LAT[1] and NT_LON[0] <= lon <= NT_LON[1], k
print(f"{len(MANUAL_OVERRIDES)} manual overrides, all inside the NT box")


23 manual overrides, all inside the NT box


## 5. ACARA site coordinates

ACARA's *School Location 2025* lists every Australian school campus with its own latitude/longitude.
Three ways a school on the NT list finds its ACARA row, tried in order:

| method | example | precision |
|---|---|---|
| `acara:exact` | same name once case, `&`/`and` and punctuation are ignored | site |
| `acara:alias:<ACARA name>` | *Sanderson High School* → *Sanderson Middle School* (Malak) | site |
| `acara:sibling:<ACARA name>` | *Alawa Pre School* → *Alawa Primary School* | sibling_school |

The alias list is short and written out by hand. Every pair is the same school in the same suburb,
listed under a newer name (NT middle schools renamed since the 2025 ACARA file) or as a campus.
No fuzzy matching: a wrong site is worse than the locality fallback.


In [6]:
acara = pd.read_excel(ACARA_XLSX, sheet_name="SchoolLocations 2025")
acara = acara[acara["State"] == "NT"].copy()


def school_key(name) -> str:
    """Normalise a school name for matching: upper-case, '&' -> AND, apostrophes dropped, punctuation -> space."""
    n = str(name).upper().replace("&", "AND")
    n = re.sub(r"[’']", "", n)
    return re.sub(r"[^A-Z0-9]+", " ", n).strip()


acara["key"] = acara["School Name"].map(school_key)
assert acara["key"].is_unique, "two NT ACARA schools normalise to the same name"
assert in_nt(acara.rename(columns={"Latitude": "latitude", "Longitude": "longitude"})).all()
ACARA = acara.set_index("key")[["School Name", "Suburb", "Latitude", "Longitude"]]

# NT list name -> ACARA name. Same school, same suburb, different label.
ACARA_ALIASES = {
    "Dripstone Secondary College": "Dripstone Middle School",                     # Tiwi
    "Nightcliff High School": "Nightcliff Middle School",                         # Nightcliff
    "Sanderson High School": "Sanderson Middle School",                           # Malak
    "Australian International Islamic College": "Australian International Islamic College Darwin",  # Berrimah
    "Good Shepherd - Leanyer Campus": "Good Shepherd Lutheran College - St Andrews Campus",         # Leanyer
    "Nyangatjatjara College - Docker River Campus":
        "Nyangatjatjara College - Kaltukatjara Campus (Docker River) Primary Campus",               # Kaltukatjara
    "Nyangatjatjara College - Imanpa Campus": "Nyangatjatjara College - Imanpa Primary Campus",      # Imanpa
    "Our Lady of the Sacred Heart Catholic College - Bath Street Campus":
        "Our Lady of the Sacred Heart College - Bath Street Campus",                                # Alice Springs
    "Our Lady of the Sacred Heart Catholic College - Sadadeen Campus":
        "Our Lady of the Sacred Heart College - Sadadeen Road Campus",                              # Alice Springs
    "St Joseph's Catholic Flexible Learning Centre": "St Joseph's Catholic Flexible School",        # Alice Springs
}
for alias in ACARA_ALIASES.values():
    assert school_key(alias) in ACARA.index, alias

# A stand-alone preschool shares the grounds of the school with the same name; try these in order.
PRESCHOOL_SUFFIX = " PRE SCHOOL"
SIBLING_SUFFIXES = [" PRIMARY SCHOOL", " SCHOOL", " AREA SCHOOL", " COLLEGE"]


def match_acara(name):
    """Return (lat, lon, method, precision) from ACARA, or None."""
    key = school_key(name)
    if key in ACARA.index:
        r = ACARA.loc[key]
        return r.Latitude, r.Longitude, "acara:exact", "site"
    if name in ACARA_ALIASES:
        r = ACARA.loc[school_key(ACARA_ALIASES[name])]
        return r.Latitude, r.Longitude, f"acara:alias:{r['School Name']}", "site"
    if key.endswith(PRESCHOOL_SUFFIX):
        base = key[: -len(PRESCHOOL_SUFFIX)]
        for suffix in SIBLING_SUFFIXES:
            if base + suffix in ACARA.index:
                r = ACARA.loc[base + suffix]
                return r.Latitude, r.Longitude, f"acara:sibling:{r['School Name']}", "sibling_school"
    return None


print(f"ACARA NT schools: {len(ACARA)}  |  aliases: {len(ACARA_ALIASES)}")


ACARA NT schools: 221  |  aliases: 10


## 6. Match every school

ACARA first (section 5). Only a school ACARA can't place falls through to the locality tiers,
tried in order. The first hit wins and is recorded in `school_geocode_method`:

| tier | example | precision |
|---|---|---|
| `acara:exact` / `acara:alias:` / `acara:sibling:` | see section 5 | site / sibling_school |
| `manual:<source>` | Alawa → Wikipedia | suburb |
| `exact:<gazetteer source>` | Adelaide River → medical suburb centroid | suburb / community |
| `prefix:<gazetteer source>` | *Pickertaramoor Melville Island* → Pickertaramoor | community |
| `name:bushtel` | *Nawarddeken Academy Manmoyi School* → Manmoyi | community |
| `region_fallback:<region>-><town>` | the region's main town | region (coarse!) |
| `unmatched` | no coordinate. Better than a wrong one | none |

**Fuzzy matching is deliberately not used.** Short Darwin suburb names (Alawa, Anula, Nakara)
kept fuzzy-matching remote BushTel aliases hundreds of km away. The `name:` tier only accepts a
**whole-word** BushTel community name of 5+ letters and prefers the longest match.

In [7]:
# Main town of each DoE region. Used only when nothing better matches.
# "Top End" has no single town of its own, so it's intentionally left out.
REGION_FALLBACK = {
    "DARWIN": "DARWIN", "PALMERSTON": "PALMERSTON", "CENTRAL": "ALICE SPRINGS",
    "BIG RIVERS": "KATHERINE", "EAST ARNHEM": "NHULUNBUY", "BARKLY": "TENNANT CREEK",
    "WEST ARNHEM": "JABIRU",
}
PRECISION = {"medical_suburb": "suburb", "emergency_suburb": "suburb",
             "bushtel": "community", "bushtel_alias": "community"}

# BushTel names long enough to search for inside a school's name, longest first.
NAME_KEYS = sorted((k for k in bushtel_keys if len(k) >= 5), key=len, reverse=True)


def match_locality(name, locality, region):
    """Return (lat, lon, method, precision) from the locality gazetteer."""
    key = norm(locality)
    if key:
        # Tier 1: cited manual coordinate.
        if key in MANUAL_OVERRIDES:
            lat, lon, src = MANUAL_OVERRIDES[key]
            return lat, lon, f"manual:{src}", "suburb"
        # Tier 2: exact gazetteer hit.
        if key in gaz:
            lat, lon, src = gaz[key]
            return lat, lon, f"exact:{src}", PRECISION[src]
        # Tier 3: drop trailing words ("PICKERTARAMOOR MELVILLE ISLAND" -> "PICKERTARAMOOR").
        words = key.split()
        for n in range(len(words) - 1, 0, -1):
            shorter = " ".join(words[:n])
            if len(shorter) >= 5 and shorter in gaz:
                lat, lon, src = gaz[shorter]
                return lat, lon, f"prefix:{src}", PRECISION[src]
    # Tier 4: a BushTel community named in the school's own name.
    upper_name = norm(name)
    for k in NAME_KEYS:
        if re.search(rf"\b{re.escape(k)}\b", upper_name):
            lat, lon, src = gaz[k]
            return lat, lon, "name:bushtel", "community"
    # Tier 5: coarse regional fallback.
    town = REGION_FALLBACK.get(norm(region))
    if town and town in gaz:
        lat, lon, _ = gaz[town]
        return lat, lon, f"region_fallback:{region}->{town}", "region"
    return None, None, "unmatched", "none"


def match(name, locality, region):
    """ACARA site if there is one, otherwise the locality fallback."""
    return match_acara(name) or match_locality(name, locality, region)


# The locality point is kept for every school (not written) so section 8 can check each ACARA match against it.
locality_hit = schools.apply(lambda r: match_locality(r["name"], r["school_locality"], r["school_region"]), axis=1)
matched = schools.apply(lambda r: match(r["name"], r["school_locality"], r["school_region"]), axis=1)
schools[["latitude", "longitude", "school_geocode_method", "school_geocode_precision"]] = pd.DataFrame(
    matched.tolist(), index=schools.index)

print(schools["school_geocode_precision"].value_counts().to_string(), "\n")
print(schools["school_geocode_method"].str.split(":").str[0].value_counts().to_string())


school_geocode_precision
site              200
sibling_school     56
suburb             15
community           1
region              1 

school_geocode_method
acara              256
exact               12
manual               4
region_fallback      1


### 6a. Review the matches that aren't a plain ACARA or exact locality hit

In [8]:
review = schools[~schools["school_geocode_method"].str.startswith(("acara:exact", "exact", "manual"))]
print(review[["name", "school_locality", "school_geocode_method", "latitude", "longitude"]].to_string())


                                                                   name school_locality                                                                   school_geocode_method   latitude   longitude
2                                                      Alawa Pre School           Alawa                                                      acara:sibling:Alawa Primary School -12.380220  130.877540
11                                                 Alyangula Pre School       Alyangula                                                     acara:sibling:Alyangula Area School -13.849640  136.420110
16                                                  Angurugu Pre School        Angurugu                                                           acara:sibling:Angurugu School -13.982530  136.459330
18                                                     Anula Pre School           Anula                                                      acara:sibling:Anula Primary School -12.391620  130.893690
23   

## 7. Fill a missing region from schools in the same locality

One school has no region. If every other school in its locality shares one region, that region is
used. Otherwise the gap stays.

In [9]:
locality_region = (schools.dropna(subset=["school_region"])
                   .groupby(schools["school_locality"].map(norm))["school_region"]
                   .agg(lambda s: s.iloc[0] if s.nunique() == 1 else None))
missing = schools["school_region"].isna()
schools.loc[missing, "school_region"] = schools.loc[missing, "school_locality"].map(norm).map(locality_region)
print(schools.loc[missing, ["name", "school_locality", "school_region"]].to_string())


                                                     name school_locality school_region
226  St Joseph's Catholic Flexible Learning Centre Darwin        Berrimah        Darwin


## 8. Validate

* every coordinate we assigned is inside the NT
* every ACARA point sits near the school's own locality point (a far one would mean a wrong name match)
* Katherine schools now sit in Katherine (within 5 km of the Katherine Hospital)
* no Darwin school is still on the Darwin region fallback when its suburb is known

In [10]:
has_xy = schools["latitude"].notna()
assert in_nt(schools[has_xy]).all()

# ACARA vs locality point, for every school that has both.
ACARA_VS_LOCALITY_MAX_KM = 25   # a whole town/community fits inside this; a wrong-school match would not
# Checked by hand: here the *locality* was wrong and ACARA is right, so the gap is expected.
REVIEWED_FAR = {
    "Baniyala Garrangali School": "ACARA point == BushTel BANIYALA; the list's locality is the postal town Yirrkala",
    "Nyangatjatjara College": "head campus is at Yulara; the Docker River / Imanpa campuses are separate rows",
    "Bulla Camp School": "ACARA places it at Baines, ~40 km from BushTel BULLA; ACARA is the official school register",
}
loc_lat = locality_hit.str[0].astype(float)
loc_lon = locality_hit.str[1].astype(float)
from_acara = schools["school_geocode_method"].str.startswith("acara")
both = from_acara & loc_lat.notna()
shift_km = pd.Series(haversine_km(schools["latitude"], schools["longitude"], loc_lat, loc_lon), index=schools.index)
print(f"ACARA-placed schools: {int(from_acara.sum())}/{len(schools)}  "
      f"(site {int((schools['school_geocode_precision'] == 'site').sum())}, "
      f"sibling_school {int((schools['school_geocode_precision'] == 'sibling_school').sum())})")
print(f"  ACARA point vs old locality point: median {shift_km[both].median():.1f} km, max {shift_km[both].max():.1f} km")
far = schools[both & (shift_km > ACARA_VS_LOCALITY_MAX_KM)]
print(f"  > {ACARA_VS_LOCALITY_MAX_KM} km apart: {len(far)}")
print(far.assign(shift_km=shift_km[far.index].round(1))[["name", "school_locality", "school_geocode_method", "shift_km"]].to_string())
assert set(far["name"]) <= set(REVIEWED_FAR), "an unreviewed ACARA match lands far from its locality - check the name match"

kath = schools[schools["school_locality"].map(norm) == "KATHERINE"]
hosp = med[med["name"].str.startswith("Katherine Hospital")].iloc[0]
kath_km = haversine_km(kath["latitude"], kath["longitude"], hosp.latitude, hosp.longitude)
assert (kath_km < 5).all(), kath_km
print(f"{len(kath)} Katherine schools, max {kath_km.max():.1f} km from Katherine Hospital")

print(f"geocoded {int(has_xy.sum())}/{len(schools)} schools ({has_xy.mean():.1%})")
print("still unmatched:", schools.loc[~has_xy, ["name", "school_locality_raw"]].values.tolist())
print("\nlargest groups of schools sharing one exact point (preschool + its school, or a locality fallback):")
shared = schools[has_xy].groupby(["latitude", "longitude"])["name"].agg(["size", lambda s: "; ".join(s)])
print(shared[shared["size"] > 1].sort_values("size", ascending=False).head(5).to_string(header=False))


ACARA-placed schools: 256/273  (site 200, sibling_school 56)
  ACARA point vs old locality point: median 0.5 km, max 194.5 km
  > 25 km apart: 3
                           name school_locality school_geocode_method  shift_km
26   Baniyala Garrangali School        Yirrkala           acara:exact     128.0
42            Bulla Camp School           Bulla           acara:exact      39.8
187      Nyangatjatjara College    Kaltukatjara           acara:exact     194.5
9 Katherine schools, max 4.9 km from Katherine Hospital
geocoded 273/273 schools (100.0%)
still unmatched: []

largest groups of schools sharing one exact point (preschool + its school, or a locality fallback):
latitude   longitude                                                                                                              
-23.699342 133.880146  4  Alice Springs Language Centre; Ida Standley Pre School; Nathalie Gorey Pre School; Rona Glynn Pre School
-23.708500 133.859500  2                                      

## 9. Write output

In [11]:
OUT_COLS = ["source", "name", "latitude", "longitude", "school_locality", "school_geocode_method",
            "school_geocode_precision", "school_region", "school_sector", "school_remoteness",
            "school_is_preschool", "school_is_primary", "school_is_middle", "school_is_senior",
            "school_is_faft"]
schools[OUT_COLS].to_csv(PROCESSED / "school_services.csv", index=False)
print(f"wrote school_services.csv ({len(schools)} rows)")


wrote school_services.csv (273 rows)


### Known gaps (not fixed here, on purpose)

* **Schools ACARA can't place keep locality precision.** A handful of stand-alone Alice Springs
  preschools, language centres, the hospital school, flexible-learning centres and program-only
  entries (e.g. *Family as First Teachers*) are not in ACARA 2025. Check `school_geocode_precision`.
* **`sibling_school` is not a surveyed site.** A preschool takes its same-named school's point; that is
  normally the same grounds, but not guaranteed.
* **`school_sector` is blank for ~60 rows**, all government pre-schools, and
  **`school_remoteness` is blank for non-government schools**. Both come from the source data.
  The NTG remoteness definition only covers government schools.
